In [ ]:
%load_ext ElasticNotebook

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%PandasProfile
### cell 22 ###

def get_n_grams(n_grams, top_n=10):
    grouped = train.groupby('discourse_type')['discourse_text']
    records = []
    # iterate once over each discourse_type, avoid repeated DataFrame queries and appends
    for dt, texts in tqdm(grouped, total=grouped.ngroups):
        texts_list = texts.tolist()
        vec = CountVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(n_grams, n_grams)
        ).fit(texts_list)
        bag = vec.transform(texts_list)
        # get a flat list of counts for each vocabulary term
        sum_counts = bag.sum(axis=0).tolist()[0]
        vocab = vec.vocabulary_
        # build (word, count) tuples and pick top_n directly in Python
        top_items = sorted(
            ((word, sum_counts[idx]) for word, idx in vocab.items()),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]
        # accumulate results in a list of tuples
        records.extend((dt, word, count) for word, count in top_items)
    # construct a single DataFrame at the end
    return pd.DataFrame(records, columns=['Discourse_type', 'words', 'counts'])

bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()